# Step 3: Trainable Delay Parameters


Replaces fixed delay taps with trainable (soft/interpolated) delay parameters.


In [ ]:

from pathlib import Path
import json
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import snntorch as snn

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print("DEVICE:", DEVICE)


def save_fig(plot_dir: Path, name: str):
    path = plot_dir / name
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print("Saved plot:", path)


def confusion_matrix_plot(y_true, y_pred, class_labels, plot_dir: Path, filename: str):
    n = len(class_labels)
    cm = np.zeros((n, n), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    im0 = axes[0].imshow(cm, cmap="Blues", aspect="auto")
    axes[0].set_title("Confusion (counts)")
    axes[0].set_xlabel("Pred")
    axes[0].set_ylabel("True")
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(cm_norm, cmap="Blues", vmin=0.0, vmax=1.0, aspect="auto")
    axes[1].set_title("Confusion (row-normalized)")
    axes[1].set_xlabel("Pred")
    axes[1].set_ylabel("True")
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    tick_idx = np.arange(n)
    for ax in axes:
        ax.set_xticks(tick_idx)
        ax.set_yticks(tick_idx)
        ax.set_xticklabels(class_labels, rotation=90)
        ax.set_yticklabels(class_labels)

    plt.tight_layout()
    save_fig(plot_dir, filename)
    plt.show()

# Step 3: Trainable delay parameters (soft/interpolated delay bank)
ART = Path("artifacts_steps/step3_trainable_delays")
PLOT_DIR = ART / "plots"
CKPT = ART / "model_step3.pt"
METRICS = ART / "metrics_step3.json"
for d in [ART, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DT = 1e-4
T_STEPS = 768
TIME = np.arange(T_STEPS) * DT
SPEED_OF_SOUND = 343.0
DIST_MIN = 0.5
DIST_MAX = 10.0
NUM_CLASSES = 20
DIST_BINS = np.linspace(DIST_MIN, DIST_MAX, NUM_CLASSES + 1)
DIST_CENTERS = 0.5 * (DIST_BINS[:-1] + DIST_BINS[1:])
N_SAMPLES = 1200
BATCH = 64
EPOCHS = 25

PULSE_CENTER = 35
PULSE_STD = 6.0
ECHO_ATT = 0.55
INIT_DELAYS = torch.linspace(0, 640, 32)


def gaussian_pulse():
    idx = np.arange(T_STEPS)
    return np.exp(-0.5 * ((idx - PULSE_CENTER) / PULSE_STD) ** 2).astype(np.float32)


def make_sample(distance_m):
    tx = gaussian_pulse()
    delay_steps = int(round((2.0 * distance_m / SPEED_OF_SOUND) / DT))
    rx = np.zeros_like(tx)
    if delay_steps < T_STEPS:
        rx[delay_steps:] = ECHO_ATT * tx[: T_STEPS - delay_steps]
    return tx.astype(np.float32), rx.astype(np.float32)


def distance_to_bin(d):
    return int(np.digitize(d, DIST_BINS[1:-1], right=False))


def build_data(seed=SEED):
    rng = np.random.default_rng(seed)
    dists = rng.uniform(DIST_MIN, DIST_MAX, size=N_SAMPLES)
    txs, rxs, ys = [], [], []
    for d in dists:
        tx, rx = make_sample(float(d))
        txs.append(tx); rxs.append(rx); ys.append(distance_to_bin(d))
    txs = np.stack(txs); rxs = np.stack(rxs); ys = np.array(ys, dtype=np.int64)
    perm = rng.permutation(len(ys))
    txs, rxs, ys = txs[perm], rxs[perm], ys[perm]
    split = int(0.8 * len(ys))
    return txs[:split], rxs[:split], ys[:split], txs[split:], rxs[split:], ys[split:]


class TrainableDelayBank(nn.Module):
    def __init__(self, init_delays, max_delay):
        super().__init__()
        self.max_delay = float(max_delay)
        init = torch.clamp(init_delays / self.max_delay, 1e-4, 1 - 1e-4)
        raw = torch.log(init / (1 - init))
        self.raw_delays = nn.Parameter(raw)

    def delays(self):
        return self.max_delay * torch.sigmoid(self.raw_delays)

    def forward(self, x):
        # x: [T,B,1]
        x_flat = x[:, :, 0]                       # [T,B]
        T, B = x_flat.shape
        D = self.raw_delays.numel()
        pad_len = int(self.max_delay) + 2
        x_pad = F.pad(x_flat, (0, 0, pad_len, 0))  # [T+pad, B]

        t = torch.arange(T, device=x.device, dtype=torch.float32)[:, None] + pad_len
        d = self.delays()[None, :]                # [1,D]
        idx_f = t - d                             # [T,D]

        idx0 = torch.floor(idx_f).long().clamp(0, x_pad.shape[0]-1)
        idx1 = (idx0 + 1).clamp(0, x_pad.shape[0]-1)
        w = (idx_f - idx0.float()).unsqueeze(-1)  # [T,D,1]

        x0 = x_pad[idx0]                          # [T,D,B]
        x1 = x_pad[idx1]                          # [T,D,B]
        out = ((1.0 - w) * x0 + w * x1).permute(0, 2, 1).contiguous()  # [T,B,D]
        return out


class CoincidenceSNNTrainableDelay(nn.Module):
    def __init__(self, init_delays, hidden_size=96, beta=0.95):
        super().__init__()
        self.delay = TrainableDelayBank(init_delays, max_delay=640)
        self.fc1 = nn.Linear(len(init_delays), hidden_size)
        self.lif1 = snn.Leaky(beta=beta)
        self.fc2 = nn.Linear(hidden_size, NUM_CLASSES)
        self.lif2 = snn.Leaky(beta=beta)

    def forward(self, tx, rx):
        dtx = self.delay(tx)
        coinc = dtx * rx
        self.lif1.reset_mem(); self.lif2.reset_mem()
        mem1=None; mem2=None; out=[]
        for t in range(coinc.shape[0]):
            s1,mem1=self.lif1(self.fc1(coinc[t]), mem1)
            s2,mem2=self.lif2(self.fc2(s1), mem2)
            out.append(s2)
        return torch.stack(out,dim=0)


def eval_model(model, loader, criterion):
    model.eval(); total_loss=0.0; cor=0; tot=0; yt=[]; yp=[]
    with torch.no_grad():
        for txb,rxb,yb in loader:
            txb=txb.to(DEVICE); rxb=rxb.to(DEVICE); yb=yb.to(DEVICE)
            tx_t=txb.unsqueeze(-1).permute(1,0,2)
            rx_t=rxb.unsqueeze(-1).permute(1,0,2)
            spk=model(tx_t,rx_t); cnt=spk.sum(dim=0)
            loss=criterion(cnt,yb); total_loss += loss.item()*yb.size(0)
            pred=cnt.argmax(dim=1)
            cor += (pred==yb).sum().item(); tot += yb.size(0)
            yt.append(yb.cpu().numpy()); yp.append(pred.cpu().numpy())
    return total_loss/tot, cor/tot, np.concatenate(yt), np.concatenate(yp)


Xtx_tr,Xrx_tr,y_tr,Xtx_te,Xrx_te,y_te = build_data()
train_ds=TensorDataset(torch.from_numpy(Xtx_tr), torch.from_numpy(Xrx_tr), torch.from_numpy(y_tr))
test_ds=TensorDataset(torch.from_numpy(Xtx_te), torch.from_numpy(Xrx_te), torch.from_numpy(y_te))
train_loader=DataLoader(train_ds,batch_size=BATCH,shuffle=True)
test_loader=DataLoader(test_ds,batch_size=BATCH,shuffle=False)

model=CoincidenceSNNTrainableDelay(INIT_DELAYS, hidden_size=96, beta=0.95).to(DEVICE)
crit=nn.CrossEntropyLoss(); opt=torch.optim.Adam(model.parameters(), lr=1e-3)

hist={"train_loss":[],"test_loss":[],"train_acc":[],"test_acc":[]}
best=None; best_test=float("inf")

for ep in range(1,EPOCHS+1):
    model.train(); run_loss=0.0; run_cor=0; run_tot=0
    for txb,rxb,yb in train_loader:
        txb=txb.to(DEVICE); rxb=rxb.to(DEVICE); yb=yb.to(DEVICE)
        tx_t=txb.unsqueeze(-1).permute(1,0,2)
        rx_t=rxb.unsqueeze(-1).permute(1,0,2)
        opt.zero_grad(); spk=model(tx_t,rx_t); cnt=spk.sum(dim=0)
        loss=crit(cnt,yb)
        # light regularization to keep delays spread/within useful region
        delays = model.delay.delays()
        reg = 1e-5 * ((delays[1:] - delays[:-1]).abs().mean())
        total = loss + reg
        total.backward(); opt.step()
        run_loss += loss.item()*yb.size(0)
        pred=cnt.argmax(dim=1); run_cor += (pred==yb).sum().item(); run_tot += yb.size(0)
    tr_loss=run_loss/run_tot; tr_acc=run_cor/run_tot
    te_loss,te_acc,y_true,y_pred = eval_model(model,test_loader,crit)
    hist["train_loss"].append(tr_loss); hist["test_loss"].append(te_loss)
    hist["train_acc"].append(tr_acc); hist["test_acc"].append(te_acc)
    if te_loss < best_test:
        best_test=te_loss; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    if ep==1 or ep%5==0 or ep==EPOCHS:
        print(f"Epoch {ep:02d}/{EPOCHS} | train_loss={tr_loss:.4f}, train_acc={tr_acc*100:.1f}% | test_loss={te_loss:.4f}, test_acc={te_acc*100:.1f}%")

model.load_state_dict(best)
torch.save({"model_state_dict":model.state_dict(),"history":hist}, CKPT)
print("Saved checkpoint:", CKPT)

# curves
epx=np.arange(1,EPOCHS+1)
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
ax[0].plot(epx,hist["train_loss"],label="train"); ax[0].plot(epx,hist["test_loss"],label="test"); ax[0].set_title("Loss"); ax[0].grid(alpha=0.3); ax[0].legend()
ax[1].plot(epx,np.array(hist["train_acc"])*100,label="train"); ax[1].plot(epx,np.array(hist["test_acc"])*100,label="test"); ax[1].set_title("Accuracy (%)"); ax[1].grid(alpha=0.3); ax[1].legend()
plt.tight_layout(); save_fig(PLOT_DIR,"curves_step3.png"); plt.show()

# learned delays plot
learned = model.delay.delays().detach().cpu().numpy()
plt.figure(figsize=(8,4))
plt.plot(np.sort(learned), marker='o')
plt.title("Learned trainable delays (sorted)")
plt.xlabel("Delay index (sorted)")
plt.ylabel("Delay (steps)")
plt.grid(alpha=0.3)
plt.tight_layout(); save_fig(PLOT_DIR,"learned_delays_step3.png"); plt.show()

te_loss,te_acc,y_true,y_pred = eval_model(model,test_loader,crit)
confusion_matrix_plot(y_true,y_pred,[f"{d:.1f}" for d in DIST_CENTERS],PLOT_DIR,"confusion_step3.png")

with open(METRICS,"w",encoding="utf-8") as f:
    json.dump({"best_test_loss":best_test,"final_test_acc":float(te_acc),"final_train_acc":float(hist["train_acc"][-1]),"learned_delay_min":float(np.min(learned)),"learned_delay_max":float(np.max(learned))},f,indent=2)
print("Saved metrics:",METRICS)
